In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

# Inicializa a sessão do Spark (caso ainda não esteja rodando)
spark = SparkSession.builder.appName("Exercicio2_Clima").getOrCreate()

# 1. Trata os valores "NA" no dataset na hora da leitura
df_spark = spark.read.option("header", "true") \
                     .option("nullValue", "NA") \
                     .csv("clima.csv")

# 2. Converte os campos Umidade9 e Temperatura9 para double
df_tratado = df_spark.withColumn("Umidade9", col("Umidade9").cast("double")) \
                     .withColumn("Temperatura9", col("Temperatura9").cast("double"))

# 3. Cria a nova coluna SituacaoClimatica com as regras condicionais[cite: 4]
df_condicoes = df_tratado.withColumn(
    "SituacaoClimatica",
    when((col("Temperatura9") > 30) & (col("Umidade9") < 40), "Quente e Seco") \
    .when((col("Temperatura9") > 25) & (col("Umidade9") >= 40), "Quente e Úmido") \
    .otherwise("Clima Ameno")
)

# 5. Filtra apenas as localidades onde: Temperatura9 > 20[cite: 4]
df_filtrado = df_condicoes.filter(col("Temperatura9") > 20)

# 4. Exibe apenas: Localizacao, Temperatura9, Umidade9 e SituacaoClimatica[cite: 4]
df_final = df_filtrado.select("Localizacao", "Temperatura9", "Umidade9", "SituacaoClimatica")

# Mostra o resultado na tela
df_final.show(truncate=False)